# 01 — Pipeline minimal: from file to fields in 5 minutes

This is the shortest path through `idp.Pipeline`. You'll see the same flow
that `examples/01_pipeline_minimal.py` runs, but as cells you can re-run
one at a time and tweak.

**No API key required** — we use the in-tree `MockBackend` so the pipeline
actually runs end-to-end. Real backends (openai, anthropic, china:*) return
valid schemas and validate cleanly; the mock returns empty defaults so we
can see the full six-stage pipeline execute.

The six stages: **ingest → parse → classify → extract → assess → validate**.


In [1]:
import logging
logging.getLogger("idp").setLevel(logging.CRITICAL)

from idp.console import pretty_print_result
from idp.core.document import Document
from idp.core.schemas import Invoice
from idp.llm.backend import get_backend
from idp.pipeline.pipeline import Pipeline

# Resolve sample path relative to the notebook location so cells work
# from any CWD (notebooks are often opened from a different directory).
import os
from pathlib import Path
_REPO_ROOT = Path(os.environ.get("IDP_REPO_ROOT", Path.cwd()))
SAMPLE = _REPO_ROOT / "src/idp/eval/datasets/invoices/docs/inv-001.txt"

## 1. Load a sample document

The repo ships three sample invoices under `src/idp/eval/datasets/invoices/docs/`.
`Document.from_path()` reads the file and produces a `Document` object that
the pipeline can consume. No parsing yet — that happens in stage 2.


In [2]:
doc = Document.from_path(SAMPLE)
print(f"doc_id:   {doc.doc_id}")
print(f"path:     {doc.source_path}")
print(f"ext:      {doc.extension}")
print(f"preview:  {doc.raw_text[:120]!r}...")

doc_id:   inv-001-98edfe6954774ae6
path:     /Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt
ext:      txt
preview:  ''...


## 2. Build a pipeline

A `Pipeline` is bound to one schema (the contract) and one backend (the LLM).
For this first run, mock backend + Invoice schema. The mock returns empty
strings/zeros so we can see every stage of the pipeline execute end-to-end.


In [3]:
backend = get_backend("mock")
pipeline = Pipeline(backend=backend, schema=Invoice)
print(f"backend: {pipeline.backend_name}")
print(f"schema:  {pipeline.schema.__name__}")

backend: mock
schema:  Invoice


## 3. Run it

`pipeline.run(doc)` executes all six stages and returns a `PipelineResult`.
The interesting fields are `extraction`, `confidence`, `validation_passed`,
and `classification`.


In [4]:
result = pipeline.run(doc)
pretty_print_result(result)


=== /Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt ===
schema:    Invoice
backend:   mock (ocr_llm)
classify:  invoice (conf=0.99)
validate:  FAIL
timings:   parse=0.000s, classify=0.000s, route=0.000s, extract=0.056s, assess=0.000s, validate=0.000s

extraction:
{
  "invoice_number": "",
  "vendor_name": "",
  "total_amount": 0.0,
  "invoice_date": {},
  "due_date": {},
  "vendor_address": {},
  "customer_name": {},
  "customer_address": {},
  "subtotal": {},
  "tax_amount": {},
  "currency": {},
  "line_items": [
    null
  ]
}

confidence (ascending):
  invoice_number           0.10 [REVIEW]
  vendor_name              0.10 [REVIEW]
  invoice_date             0.10 [REVIEW]
  due_date                 0.10 [REVIEW]
  vendor_address           0.10 [REVIEW]
  customer_name            0.10 [REVIEW]
  customer_address         0.10 [REVIEW]
  subtotal                 0.10 [REVIEW]
  tax_amount               0.10 [REVIEW]
  currency                 0.10 [REVIEW]
  line_

## 4. Read the result fields directly

If you want a Python dict (instead of the pretty-printed view), read
`result.document.extraction`. Confidence and validation are also available
as plain dicts.


In [5]:
print("extraction:", result.document.extraction)
print("confidence:", result.confidence)
print("validation_passed:", result.validation_passed)
print("classification:", result.classification)
print("total_seconds:", round(sum(t.seconds for t in result.timings), 3))
print("per_stage:", [(t.name, round(t.seconds, 3)) for t in result.timings])

extraction: {'invoice_number': '', 'vendor_name': '', 'total_amount': 0.0, 'invoice_date': {}, 'due_date': {}, 'vendor_address': {}, 'customer_name': {}, 'customer_address': {}, 'subtotal': {}, 'tax_amount': {}, 'currency': {}, 'line_items': [None]}
confidence: {'invoice_number': 0.1, 'vendor_name': 0.1, 'total_amount': 0.7, 'invoice_date': 0.1, 'due_date': 0.1, 'vendor_address': 0.1, 'customer_name': 0.1, 'customer_address': 0.1, 'subtotal': 0.1, 'tax_amount': 0.1, 'currency': 0.1, 'line_items': 0.6499999999999999}
validation_passed: False
classification: invoice
total_seconds: 0.057
per_stage: [('parse', 0.0), ('classify', 0.0), ('route', 0.0), ('extract', 0.056), ('assess', 0.0), ('validate', 0.0)]


## 5. Try with a real backend

The mock is for testing the pipeline shape. To see real extraction, set an
API key in your shell (or in this cell) and rerun:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # or ANTHROPIC_API_KEY, or IDP_BACKEND=ollama
backend = get_backend("auto")  # picks the right backend from env
pipeline = Pipeline(backend=backend, schema=Invoice)
result = pipeline.run(doc)
pretty_print_result(result)
```

You'll see `extraction` populated with real fields and `validation_passed: True`.

## What's next

- **Notebook 02** — the HITL loop: low-confidence fields, human review,
  policy override, second run shows the override path.
- **Notebook 03** — batch processing with `process_batch()`, checkpoint
  resume, and the `BatchItemResult` workflow.
